In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import copy

from sklearn.linear_model import LogisticRegression,ElasticNet
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
# 评价方法
from sklearn.metrics import roc_curve, roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split,StratifiedKFold

import lightgbm as lgb
#from catboost import CatBoostClassifier
from tqdm import tqdm
import time 
import datetime

In [2]:
fea_df = pd.read_pickle('fea_df2.pkl')

In [3]:
data=fea_df

In [4]:
#candle_begin_time,symbol,is_spot,close
TIME='candle_begin_time'
TARGET='Return_24'
USE_COLS = data.columns.difference(['candle_begin_time','symbol','is_spot','close','Return_24'])

In [5]:
data['count'] = data.groupby('candle_begin_time')['symbol'].transform('count') - 1

In [6]:
# data.groupby('candle_begin_time')[USE_COLS].rank(method='min', ascending=True) / data['count']

In [7]:
data['count'] = data.groupby('candle_begin_time')['symbol'].transform('count') - 1
for fea in tqdm(list(USE_COLS)):
    # 对每个特征按 'candle_begin_time' 分组，计算排名和归一化
    data[fea] = (data.groupby('candle_begin_time')[fea].rank(method='min', ascending=True)-1) / data['count']

100%|██████████| 73/73 [02:32<00:00,  2.08s/it]


In [8]:
# 任务
# lgb 划分训练集 验证集，在验证集上早停
# 验证集分别选择 2021 2022 2023 2024
# 统计 验证集 的 rank_ic 早停轮次  其他metric

In [9]:
for fea in USE_COLS:
    if data[fea].dtypes == 'object':
        data[fea] = data[fea].astype('float')
        print(f'{fea}已转换!')

In [10]:
test = data[pd.to_datetime(data[TIME]) >= pd.to_datetime('2024-01-01')].reset_index(drop=True)  # 测试集
train = data[pd.to_datetime(data[TIME]) < pd.to_datetime('2024-01-01')].reset_index(drop=True)  # 训练集

In [11]:
X_train = train[USE_COLS]
y_train = train[TARGET]
X_val = test[USE_COLS]
y_val = test[TARGET]

In [12]:
t=time.time()
lgb_train = lgb.Dataset(X_train, label = y_train)
lgb_eval = lgb.Dataset(X_val, label = y_val)
params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',  # 将目标设置为回归
    'metric': 'rmse',  # 使用L2误差作为评估指标，或者你可以选择 'mae' 或 'rmse'
    'num_leaves': 128,
    'max_depth': 20,
    'min_data_in_leaf': 50,
    'learning_rate': 0.01,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 1,
    'lambda_l2': 1,  # 正则化，越小正则化效果越强
    'min_gain_to_split': 0.2,
    'verbose': -1,  # 关闭日志输出
    # 'device': 'gpu',  # 如果需要使用 GPU 训练，取消注释
}

# 训练回归模型
lgb_clf = lgb.train(
    params,
    lgb_train,
    num_boost_round=10000,
    valid_sets=lgb_eval,
    callbacks=[
        lgb.early_stopping(stopping_rounds=200),  # 提前停止
        lgb.log_evaluation(period=50)  # 每 50 次迭代打印一次日志
    ]
)
print(time.time()-t)
preds = lgb_clf.predict(X_val)
test['preds']=preds
hourly_columns= test.groupby('candle_begin_time')[['preds', 'Return_24']].corr(method='spearman').unstack().iloc[:, 1]
hourly_columns.mean()

Training until validation scores don't improve for 200 rounds
[50]	valid_0's rmse: 0.0636294
[100]	valid_0's rmse: 0.0636416
[150]	valid_0's rmse: 0.0636557
[200]	valid_0's rmse: 0.0636648
Early stopping, best iteration is:
[2]	valid_0's rmse: 0.0636145
342.9588997364044


np.float64(0.04989912244439051)

In [13]:
print('1')

1


In [14]:
test = data[pd.to_datetime(data[TIME]) < pd.to_datetime('2022-01-01')].reset_index(drop=True)  # 测试集
train = data[pd.to_datetime(data[TIME]) >= pd.to_datetime('2022-01-01')].reset_index(drop=True)  # 训练集
X_train = train[USE_COLS]
y_train = train[TARGET]
X_val = test[USE_COLS]
y_val = test[TARGET]
t=time.time()
lgb_train = lgb.Dataset(X_train, label = y_train)
lgb_eval = lgb.Dataset(X_val, label = y_val)
params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',  # 将目标设置为回归
    'metric': 'rmse',  # 使用L2误差作为评估指标，或者你可以选择 'mae' 或 'rmse'
    'num_leaves': 128,
    'max_depth': 20,
    'min_data_in_leaf': 50,
    'learning_rate': 0.01,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 1,
    'lambda_l2': 1,  # 正则化，越小正则化效果越强
    'min_gain_to_split': 0.2,
    'verbose': -1,  # 关闭日志输出
    # 'device': 'gpu',  # 如果需要使用 GPU 训练，取消注释
}

# 训练回归模型
lgb_clf = lgb.train(
    params,
    lgb_train,
    num_boost_round=10000,
    valid_sets=lgb_eval,
    callbacks=[
        lgb.early_stopping(stopping_rounds=200),  # 提前停止
        lgb.log_evaluation(period=50)  # 每 50 次迭代打印一次日志
    ]
)
print(time.time()-t)
preds = lgb_clf.predict(X_val)
test['preds']=preds
hourly_columns= test.groupby('candle_begin_time')[['preds', 'Return_24']].corr(method='spearman').unstack().iloc[:, 1]
hourly_columns.mean()

Training until validation scores don't improve for 200 rounds
[50]	valid_0's rmse: 0.0894888
[100]	valid_0's rmse: 0.0894827
[150]	valid_0's rmse: 0.0894818
[200]	valid_0's rmse: 0.0894809
[250]	valid_0's rmse: 0.0894805
[300]	valid_0's rmse: 0.0894805
[350]	valid_0's rmse: 0.0894805
[400]	valid_0's rmse: 0.0894805
Early stopping, best iteration is:
[220]	valid_0's rmse: 0.0894805
529.3599829673767


np.float64(0.05289657379619865)

In [15]:
# 转换 TIME 列为 datetime 类型（如果数据中的 TIME 列还不是 datetime 类型的话）
data[TIME] = pd.to_datetime(data[TIME])
# 选择2022年整年的数据作为测试集
test = data[(data[TIME] >= pd.to_datetime('2022-01-01')) & (data[TIME] < pd.to_datetime('2023-01-01'))].reset_index(drop=True)
# 其余的数据作为训练集
train = data[~((data[TIME] >= pd.to_datetime('2022-01-01')) & (data[TIME] < pd.to_datetime('2023-01-01')))].reset_index(drop=True)

In [16]:
X_train = train[USE_COLS]
y_train = train[TARGET]
X_val = test[USE_COLS]
y_val = test[TARGET]
t=time.time()
lgb_train = lgb.Dataset(X_train, label = y_train)
lgb_eval = lgb.Dataset(X_val, label = y_val)
params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',  # 将目标设置为回归
    'metric': 'rmse',  # 使用L2误差作为评估指标，或者你可以选择 'mae' 或 'rmse'
    'num_leaves': 128,
    'max_depth': 20,
    'min_data_in_leaf': 50,
    'learning_rate': 0.01,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 1,
    'lambda_l2': 1,  # 正则化，越小正则化效果越强
    'min_gain_to_split': 0.2,
    'verbose': -1,  # 关闭日志输出
    # 'device': 'gpu',  # 如果需要使用 GPU 训练，取消注释
}

# 训练回归模型
lgb_clf = lgb.train(
    params,
    lgb_train,
    num_boost_round=10000,
    valid_sets=lgb_eval,
    callbacks=[
        lgb.early_stopping(stopping_rounds=200),  # 提前停止
        lgb.log_evaluation(period=50)  # 每 50 次迭代打印一次日志
    ]
)
print(time.time()-t)
preds = lgb_clf.predict(X_val)
test['preds']=preds
hourly_columns= test.groupby('candle_begin_time')[['preds', 'Return_24']].corr(method='spearman').unstack().iloc[:, 1]
hourly_columns.mean()

Training until validation scores don't improve for 200 rounds
[50]	valid_0's rmse: 0.0643937
[100]	valid_0's rmse: 0.0644022
[150]	valid_0's rmse: 0.0644093
[200]	valid_0's rmse: 0.0644104
Early stopping, best iteration is:
[47]	valid_0's rmse: 0.0643929
467.2673280239105


np.float64(0.06699586363175034)

In [17]:
# 转换 TIME 列为 datetime 类型（如果数据中的 TIME 列还不是 datetime 类型的话）
data[TIME] = pd.to_datetime(data[TIME])
# 选择2022年整年的数据作为测试集
test = data[(data[TIME] >= pd.to_datetime('2023-01-01')) & (data[TIME] < pd.to_datetime('2024-01-01'))].reset_index(drop=True)
# 其余的数据作为训练集
train = data[~((data[TIME] >= pd.to_datetime('2023-01-01')) & (data[TIME] < pd.to_datetime('2024-01-01')))].reset_index(drop=True)
X_train = train[USE_COLS]
y_train = train[TARGET]
X_val = test[USE_COLS]
y_val = test[TARGET]
t=time.time()
lgb_train = lgb.Dataset(X_train, label = y_train)
lgb_eval = lgb.Dataset(X_val, label = y_val)
params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',  # 将目标设置为回归
    'metric': 'rmse',  # 使用L2误差作为评估指标，或者你可以选择 'mae' 或 'rmse'
    'num_leaves': 128,
    'max_depth': 20,
    'min_data_in_leaf': 50,
    'learning_rate': 0.01,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 1,
    'lambda_l2': 1,  # 正则化，越小正则化效果越强
    'min_gain_to_split': 0.2,
    'verbose': -1,  # 关闭日志输出
    # 'device': 'gpu',  # 如果需要使用 GPU 训练，取消注释
}

# 训练回归模型
lgb_clf = lgb.train(
    params,
    lgb_train,
    num_boost_round=10000,
    valid_sets=lgb_eval,
    callbacks=[
        lgb.early_stopping(stopping_rounds=200),  # 提前停止
        lgb.log_evaluation(period=50)  # 每 50 次迭代打印一次日志
    ]
)
print(time.time()-t)
preds = lgb_clf.predict(X_val)
test['preds']=preds
hourly_columns= test.groupby('candle_begin_time')[['preds', 'Return_24']].corr(method='spearman').unstack().iloc[:, 1]
hourly_columns.mean()

Training until validation scores don't improve for 200 rounds
[50]	valid_0's rmse: 0.0496304
[100]	valid_0's rmse: 0.0496576
[150]	valid_0's rmse: 0.0496813
[200]	valid_0's rmse: 0.0497116
Early stopping, best iteration is:
[3]	valid_0's rmse: 0.0496068
410.34375977516174


np.float64(0.04102993507137193)

In [21]:
fea_df = pd.read_pickle('fea_df2.pkl')
test['origin_Return_24'] = data[(data[TIME] >= pd.to_datetime('2023-01-01')) & (data[TIME] < pd.to_datetime('2024-01-01'))]['Return_24'].values
hourly_columns= test.groupby('candle_begin_time')[['preds', 'origin_Return_24']].corr(method='spearman').unstack().iloc[:, 1]
hourly_columns.mean()

np.float64(0.04102993507137193)